### This is an alarm system based on motion detection . 
- It uses black and white visualisations to detect movement where black is things that havent changed and white is things that have moved. So the more the white the alarm will be trickered 
- This can be used to monitor areas 

In [3]:
##importing libraries 
import cv2
import imutils
import threading 
##import winsound ##this is a windows library for sound and is most likely not to work on mac or linux 


In [6]:
##setting up the camera 
capture = cv2.VideoCapture(0) ##since i have only one camera

capture.set(cv2.CAP_PROP_FRAME_WIDTH,640)
capture.set(cv2.CAP_PROP_FRAME_HEIGHT,480)
##if something exceeds these boundaries ,  they will trigger the alarm 

_, start_frame = capture.read()

start_frame = imutils.resize(start_frame, width = 500)
start_frame = cv2.cvtColor(start_frame, cv2.COLOR_BGR2GRAY)
start_frame = cv2.GaussianBlur(start_frame, (21,21) , 0)

#### setting the alarm 

In [7]:
alarm = False #this variable will tell us if an alarm is active 
alarm_mode = False ##this is used to switch to an alarm mode 
alarm_counter = 0 ##how long do we want to have moveement to trigger an alarm

def beep_alarm(): ##here we define whatever we want to happen when the alarm is triggered 
    global alarm 
    for _ in range(5):
        if not alarm_mode:
            break ##only call the function when in alarm mode and be able to terminate the alarm when in alarm mode 
        print("ALLLAAAAARRRMMMMMMMM")
    alarm = False

#### main logic for the program 

In [ ]:
while True:
    _, frame = capture.read()
    frame = imutils.resize(frame, width=500)

    if alarm_mode :
        frame_bw = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
        frame_bw =cv2.GaussianBlur(frame_bw, (5,5), 0)


        difference = cv2.absdiff(frame_bw, start_frame) ##this is the absolute difference between the footages to determine if something in the frame is moving 

        threshold = cv2.threshold(difference,25,255,cv2.THRESH_BINARY)[1]##this is the bar which when crossed we get an alarm 
        start_frame = frame_bw

        ##take the threshold and find the sum
        if threshold.sum() > 200: ##the lower the more sensitive .It will turn on at the slightest movement
            alarm_counter += 1
        else:
            if alarm_counter > 0:
                alarm_counter -= 1
        cv2.imshow("Cam",threshold)
    
    else:
        cv2.imshow("Cam",frame)

    if alarm_counter > 20: 
        if not alarm:
            alarm = True
            threading.Thread(target=beep_alarm).start()
    

    key_pressed = cv2.waitKey(30)

    if key_pressed == ord("t"):
        alarm_mode = not alarm_mode
        alarm_counter = 0 
    if key_pressed == ord("q"):
        alarm_mode = False
        break

capture.release()
cv2.destroyAllWindows()

